# Laboratorio 1: Dinamica y Estabilidad Numerica en el Modelo Logistico 1D

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nabw/UNA-taller/blob/main/notebooks/estudiante/01_modelo_logistico.ipynb)


## 1. Introduccion y Contexto Matematico

El modelo logistico, propuesto originalmente por Pierre Francois Verhulst en 1838, describe la dinamica temporal de una poblacion en un entorno con recursos finitos. En su formulacion normalizada, la fraccion de la poblacion respecto a la capacidad de carga, $x(t) \in [0, 1]$, satisface la ecuacion diferencial ordinaria (EDO):

$$\frac{dx}{dt} = C x (1 - x)$$

donde:
- $x(t)$ es la variable de estado adimensional.
- $C > 0$ representa la tasa intrinseca de crecimiento per capita.
- El termino $C x$ modela la expansion exponencial no restringida.
- El factor $(1 - x)$ modela la saturacion por competencia ambiental (densodependencia).

### Puntos de Equilibrio y Comportamiento Asintotico

Los puntos criticos corresponden a las raices de $f(x) = C x (1 - x) = 0$:
- $x^* = 0$: punto de equilibrio inestable ($f'(0) = C > 0$).
- $x^* = 1$: punto de equilibrio asintoticamente estable ($f'(1) = -C < 0$).

Para cualquier condicion inicial $x_0 \in (0, 1)$, la solucion continua es estrictamente creciente, acotada superiormente por 1 y libre de oscilaciones.

## Objetivos de Aprendizaje

1. Deducir e implementar la solucion analitica exacta en forma cerrada.
2. Implementar e integrar la trayectoria mediante el metodo de Euler explicito.
3. Implementar el metodo de Euler implicito resolviendo analiticamente la raiz cuadratica admisible.
4. Implementar el esquema Implicito-Explicito (IMEX) linealizado.
5. Comparar la estabilidad numerica en dos regimenes temporales: paso fino ($\Delta t = 0.1$) frente a paso critico ($\Delta t = 1.3$) con $C = 2.0$.


In [ ]:
# Celda de configuracion y librerias cientificas
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
import numpy as np
import matplotlib.pyplot as plt

# Estilo sobrio y legible para figuras cientificas
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "figure.titlesize": 13,
    "lines.linewidth": 2.0,
    "grid.alpha": 0.4
})

print("Librerias importadas correctamente.")


## 2. Solucion Analitica Exacta

La ecuacion logistica es separable:

$$\frac{dx}{x (1 - x)} = C dt$$

Descomponiendo en fracciones parciales:

$$\int \left( \frac{1}{x} + \frac{1}{1 - x} \right) dx = \int C dt \implies \ln\left( \frac{x}{1 - x} \right) = C t + K$$

Aplicando la condicion inicial $x(0) = x_0$ y despejando $x(t)$ de forma numericamente estable:

$$x(t) = \frac{1}{1 + \left( \frac{1 - x_0}{x_0} \right) e^{-C t}}$$


In [ ]:
def solucion_exacta(x0: float, C: float, t: np.ndarray) -> np.ndarray:
    """
    Calcula la solucion analitica exacta de la ecuacion logistica 1D.

    Parametros
    ----------
    x0 : float
        Condicion inicial x(0), con x0 en (0, 1).
    C : float
        Tasa intrinseca de crecimiento per capita (C > 0).
    t : np.ndarray
        Vector unidimensional de instantes de tiempo.

    Retorna
    -------
    np.ndarray
        Valores exactos de x(t).
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para solucion_exacta
t_test = np.array([0.0, 1.0, 5.0, 15.0])
x_test = solucion_exacta(0.1, 2.0, t_test)

assert isinstance(x_test, np.ndarray), "La funcion debe retornar un arreglo de NumPy."
assert x_test.shape == t_test.shape, "La forma del arreglo de salida debe ser igual a la de t."
assert np.isclose(x_test[0], 0.1), f"x(0) debe ser 0.1, obtenido {x_test[0]:.4f}"
assert np.isclose(x_test[-1], 1.0, atol=1e-5), f"Para t grande, x debe tender a 1.0, obtenido {x_test[-1]:.6f}"
assert np.all(np.diff(x_test) > 0), "La solucion logistica continua con x0 in (0, 1) debe ser estrictamente creciente."
print("Validacion de solucion_exacta: EXITOSA.")


## 3. Metodo de Euler Explicito

Discretizamos el intervalo $[0, T]$ en pasos regulares $t_n = n \Delta t$. Reemplazando la derivada temporal por una diferencia finita hacia adelante:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_n (1 - x_n)$$

Despejando el estado en el siguiente paso:

$$x_{n+1} = x_n + \Delta t C x_n (1 - x_n)$$

### Analisis de Estabilidad Lineal Local

Sea $x^* = 1$ el atractor asintotico. Consideremos una pequenia perturbacion $\delta x_n = x_n - 1$. Sustituyendo:

$$1 + \delta x_{n+1} = (1 + \delta x_n) + \Delta t C (1 + \delta x_n) (-\delta x_n)$$
$$\delta x_{n+1} \approx (1 - \Delta t C) \delta x_n$$

El factor de amplificacion local es $\lambda = 1 - \Delta t C$. La condicion de estabilidad lineal exige $|\lambda| < 1$:

$$-1 < 1 - \Delta t C < 1 \implies 0 < \Delta t C < 2$$

- Si $\Delta t C < 1$: convergencia monótona hacia el atractor.
- Si $1 < \Delta t C < 2$: oscilaciones amortiguadas que convergen a 1.
- Si $\Delta t C > 2$: inestabilidad lineal. Aparecen oscilaciones espurias persistentes o divergentes.


In [ ]:
def euler_explicito(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuacion logistica mediante el esquema de Euler explicito.

    Parametros
    ----------
    x0 : float
        Condicion inicial.
    C : float
        Parametro de crecimiento.
    dt : float
        Tamano del paso temporal.
    T : float
        Tiempo final.

    Retorna
    -------
    t : np.ndarray
        Malla temporal discreta.
    x : np.ndarray
        Valores discretos de la solucion.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para euler_explicito
t_exp, x_exp = euler_explicito(0.1, 2.0, 0.1, 1.0)
assert len(t_exp) == 11, f"Longitud esperada: 11, obtenida: {len(t_exp)}"
assert np.isclose(x_exp[0], 0.1), "La condicion inicial no coincide."
expected_step1 = 0.1 + 0.1 * 2.0 * 0.1 * (1.0 - 0.1)
assert np.isclose(x_exp[1], expected_step1), f"Paso 1 esperado: {expected_step1}, obtenido: {x_exp[1]}"
print("Validacion de euler_explicito: EXITOSA.")


## 4. Metodo de Euler Implicito (Backward Euler)

Evaluamos la tasa de variacion en el instante futuro $t_{n+1}$:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_{n+1} (1 - x_{n+1})$$

Multiplicando por $\Delta t$ y agrupando terminos:

$$(\Delta t C) x_{n+1}^2 + (1 - \Delta t C) x_{n+1} - x_n = 0$$

Esta es una ecuacion cuadratica de la forma $a y^2 + b y + c = 0$, con:
- $a = \Delta t C > 0$
- $b = 1 - \Delta t C$
- $c = -x_n$

El discriminante es $\Delta = b^2 - 4ac = (1 - \Delta t C)^2 + 4 \Delta t C x_n \ge 0$ para todo $x_n \ge 0$. Puesto que $c = -x_n \le 0$ y $a > 0$, el producto de las dos raices es $c/a \le 0$. Por lo tanto, existe una unica raiz real no-negativa (fisicamente admisible):

$$x_{n+1} = \frac{-(1 - \Delta t C) + \sqrt{(1 - \Delta t C)^2 + 4 \Delta t C x_n}}{2 \Delta t C}$$

Para evitar cancelacion sustractiva cuando $b > 0$, la raiz se puede formular de manera equivalente como:

$$x_{n+1} = \frac{2 x_n}{b + \sqrt{b^2 + 4 a x_n}}$$


In [ ]:
def euler_implicito(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuacion logistica mediante el esquema de Euler implicito,
    utilizando la raiz analitica positiva de la ecuacion cuadratica.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para euler_implicito
t_imp, x_imp = euler_implicito(0.1, 2.0, 0.1, 1.0)
assert len(t_imp) == 11, f"Longitud esperada: 11, obtenida: {len(t_imp)}"
assert np.isclose(x_imp[0], 0.1), "La condicion inicial no coincide."
# Verificar que satisface la relacion algebraica implicita en cada paso
residuos = x_imp[1:] - x_imp[:-1] - 0.1 * 2.0 * x_imp[1:] * (1.0 - x_imp[1:])
assert np.all(np.abs(residuos) < 1e-12), f"Residuo maximo: {np.max(np.abs(residuos)):.2e}"
print("Validacion de euler_implicito: EXITOSA.")


## 5. Esquema Implicito-Explicito (IMEX) Linealizado

Para evitar la resolucion de polinomios o metodos iterativos no lineales, podemos desacoplar la dinamica tratando el termino lineal de expansion de manera explicita y el termino no lineal de saturacion de manera implicita:

$$\frac{x_{n+1} - x_n}{\Delta t} = C x_n - C x_n x_{n+1}$$

Reorganizando la expresion algebraicamente:

$$x_{n+1} (1 + \Delta t C x_n) = x_n (1 + \Delta t C)$$

Despejando directamente $x_{n+1}$:

$$x_{n+1} = \frac{x_n (1 + \Delta t C)}{1 + \Delta t C x_n}$$

### Propiedades Analiticas Fundamentales
1. **Linealidad implícita**: El calculo de $x_{n+1}$ es una operacion aritmetica directa.
2. **Positividad incondicional**: Si $x_n \ge 0$, entonces tanto el numerador como el denominador son estrictamente positivos para cualquier $\Delta t > 0$, garantizando $x_{n+1} \ge 0$.
3. **Conservacion del punto fijo**: Si $x_n = 1$, entonces $x_{n+1} = \frac{1 + \Delta t C}{1 + \Delta t C} = 1$ de forma exacta para todo $\Delta t$.


In [ ]:
def imex_linealizado(
    x0: float,
    C: float,
    dt: float,
    T: float
) -> tuple[np.ndarray, np.ndarray]:
    """
    Resuelve la ecuacion logistica mediante el esquema IMEX linealizado.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para imex_linealizado
t_imx, x_imx = imex_linealizado(0.1, 2.0, 0.1, 1.0)
assert len(t_imx) == 11, f"Longitud esperada: 11, obtenida: {len(t_imx)}"
assert np.isclose(x_imx[0], 0.1), "La condicion inicial no coincide."
expected_p1 = (0.1 * (1.0 + 0.1 * 2.0)) / (1.0 + 0.1 * 2.0 * 0.1)
assert np.isclose(x_imx[1], expected_p1), f"Paso 1 esperado: {expected_p1}, obtenido: {x_imx[1]}"
assert np.all(x_imx >= 0.0), "El esquema IMEX debe preservar no-negatividad."
print("Validacion de imex_linealizado: EXITOSA.")


## 6. Comparacion de Regimenes Temporales: Paso Estable vs Inestable

Fijamos los parametros fisicos:
- Tasa de crecimiento: $C = 2.0$
- Condicion inicial: $x_0 = 0.05$
- Horizonte de integracion: $T = 8.0$

Evaluamos dos regimenes de paso temporal:
1. **Paso Fino (Regimen Asintotico)**: $\Delta t = 0.1$.
   $$\Delta t C = 0.2 < 2.0$$
   Dentro del radio de estabilidad lineal para todos los esquemas.

2. **Paso Critico (Regimen Inestable para Explicito)**: $\Delta t = 1.3$.
   $$\Delta t C = 2.6 > 2.0$$
   Euler explicito supera el umbral critico de estabilidad lineal, produciendo oscilaciones espurias y sobrepaso de la capacidad de carga ($x > 1$).


In [ ]:
# Parametros de simulacion
C = 2.0
x0 = 0.05
T = 8.0

dt_cases = [0.1, 1.3]
results = {}

print("=" * 75)
print(f"{'dt':>6s} | {'Metodo':>16s} | {'Error L_inf':>12s} | {'Maximo x':>10s} | {'Sobrepaso x>1':>14s}")
print("=" * 75)

for dt in dt_cases:
    results[dt] = {}
    # Referencia exacta
    t_eval = np.linspace(0.0, T, int(np.round(T / dt)) + 1)
    x_exact = solucion_exacta(x0, C, t_eval)
    results[dt]["exact"] = (t_eval, x_exact)

    # Integradores numericos
    t_exp, x_exp = euler_explicito(x0, C, dt, T)
    t_imp, x_imp = euler_implicito(x0, C, dt, T)
    t_imx, x_imx = imex_linealizado(x0, C, dt, T)

    for name, t_num, x_num in [
        ("Euler Explicito", t_exp, x_exp),
        ("Euler Implicito", t_imp, x_imp),
        ("IMEX Linealizado", t_imx, x_imx)
    ]:
        err_linf = float(np.max(np.abs(x_num - x_exact)))
        max_val = float(np.max(x_num))
        overshoot = "SI" if max_val > 1.0 + 1e-5 else "NO"
        results[dt][name] = (t_num, x_num, err_linf, max_val)
        print(f"{dt:6.2f} | {name:>16s} | {err_linf:12.4e} | {max_val:10.4f} | {overshoot:>14s}")
    print("-" * 75)

In [ ]:
# Graficas de comparacion para ambos regimenes
t_dense = np.linspace(0.0, T, 400)
x_dense = solucion_exacta(x0, C, t_dense)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Panel 1: dt = 0.1
ax = axes[0]
ax.plot(t_dense, x_dense, 'k-', label="Solucion Exacta", zorder=1)
for name, color, style in [
    ("Euler Explicito", "#d95f02", "o--"),
    ("Euler Implicito", "#7570b3", "s--"),
    ("IMEX Linealizado", "#1b9e77", "^--")
]:
    t_num, x_num, _, _ = results[0.1][name]
    ax.plot(t_num, x_num, style, color=color, markersize=4, label=name, alpha=0.85, zorder=2)
ax.set_title("Regimen Estable: dt = 0.1 (dt * C = 0.2 < 2.0)")
ax.set_xlabel("Tiempo t")
ax.set_ylabel("Densidad poblacional x(t)")
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.7)
ax.legend(frameon=True, loc="lower right")

# Panel 2: dt = 1.3
ax = axes[1]
ax.plot(t_dense, x_dense, 'k-', label="Solucion Exacta", zorder=1)
for name, color, style in [
    ("Euler Explicito", "#d95f02", "o--"),
    ("Euler Implicito", "#7570b3", "s--"),
    ("IMEX Linealizado", "#1b9e77", "^--")
]:
    t_num, x_num, _, _ = results[1.3][name]
    ax.plot(t_num, x_num, style, color=color, markersize=6, label=name, alpha=0.9, zorder=2)
ax.set_title("Regimen Inestable: dt = 1.3 (dt * C = 2.6 > 2.0)")
ax.set_xlabel("Tiempo t")
ax.axhline(1.0, color="gray", linestyle=":", alpha=0.7)
ax.legend(frameon=True, loc="lower right")

plt.tight_layout()
plt.show()


## 7. Conclusiones y Sintesis Conceptual

1. **Sensibilidad del esquema explicito al paso temporal**:
   Cuando $\Delta t C = 0.2$, el metodo de Euler explicito reproduce la curva sigmoide con precision. Sin embargo, al fijar $\Delta t = 1.3$ (donde $\Delta t C = 2.6 > 2.0$), el esquema explota localmente alrededor del punto de equilibrio $x^* = 1$, sobrepasando la cota fisica ($x \approx 1.22$) y exhibiendo oscilaciones no amortiguadas de periodo 2. Este fenomeno es puramente un artefacto de la discretizacion temporal y no representa la dinamica continua del modelo biologico.

2. **Robustez incondicional de los esquemas implicito e IMEX**:
   Tanto el metodo de Euler implicito como el esquema IMEX linealizado preservan la estabilidad cualitativa para $\Delta t = 1.3$. Ninguno de los dos presenta sobrepaso ($x \le 1.0$) ni oscilaciones espurias.

3. **Ventaja computacional del esquema IMEX**:
   El esquema IMEX logra la misma estabilidad asintotica que Euler implicito, pero sin requerir la resolucion de ecuaciones cuadraticas en cada paso, ofreciendo una actualizacion analitica directa y positiva.
